# 📊 Pipeline Preditivo de Evasão de Clientes (Churn Telecom)
### Solução Técnica de Engenharia de Dados & Machine Learning
**Perfil**: Analista de Dados & Cientista de Dados Sênior  
**Contexto**: Provedora de Serviços de Telecomunicações (Internet e Telefonia)  
**Objetivo de Negócio**: Identificar antecipadamente clientes com alto risco de cancelamento contratual (`cancelou = 1`), mitigando o custo de aquisição (CAC) e subsidiando ações preventivas de retenção via CRM/Marketing.

---
### 🛠️ Estrutura do Pipeline
1. **Fase 1: Análise Exploratória de Dados (EDA)**
2. **Fase 2: Limpeza e Tratamento de Dados (Data Prep & Diagnóstico de Outliers)**
3. **Fase 3: Engenharia de Recursos (Feature Engineering)**
4. **Fase 4: Divisão Estratificada e Balanceamento de Classes (Prevenção de Data Leakage)**
5. **Fase 5: Escalonamento Seletivo de Variáveis (StandardScaler)**
6. **Fase 6: Otimização de Hiperparâmetros e Diagnóstico de Overfitting**
7. **Fase 7: Avaliação de Métricas e Veredito Técnico de Produção**


## 0. Configuração do Ambiente e Importação de Bibliotecas


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

# Configuração estética das visualizações
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12


: 

## 1. Fase 1: Análise Exploratória de Dados (EDA)

Como analista sênior, a primeira diretriz técnica consiste em inspecionar a topologia do dataset, validar os tipos primitivos de dados, gerar as estatísticas de dispersão e tendência central e avaliar o desbalanceamento da classe alvo.

A leitura do arquivo é realizada via caminho relativo (`dataset_recuperacao_telecom.csv`), garantindo portabilidade entre ambientes de execução.


In [ ]:
# Carregamento do dataset via caminho relativo
df = pd.read_csv('dataset_recuperacao_telecom.csv')

print(f"Dimensões do Dataset: {df.shape[0]} registros por {df.shape[1]} atributos.")
print("\nTipagem dos Atributos:")
print(df.dtypes.value_counts())
print("\nResumo Geral:")
display(df.info())

print("\nResumo Estatístico Descritivo (.describe):")
display(df.describe())


### Visualizações Diagnósticas Obrigatórias da EDA:
1. **Distribuição da Variável Alvo**: verificação de assimetria de classes.
2. **Histograma de Variáveis Preditoras Contínuas**: análise de caudas e densidade amostral.
3. **Matriz de Correlação Linear de Pearson**: diagnóstico de multicolinearidade e relação direta com a evasão.


In [ ]:
fig = plt.figure(figsize=(18, 5))

# 1. Proporção da Variável Alvo (Cancelamento)
ax1 = plt.subplot(1, 3, 1)
target_counts = df['cancelou'].value_counts()
sns.barplot(x=target_counts.index, y=target_counts.values, palette=['#1f77b4', '#d62728'], ax=ax1)
ax1.set_title('Gráfico 1: Desbalanceamento do Target', fontweight='bold')
ax1.set_xlabel('Cancelou (0 = Ativo | 1 = Evasão)')
ax1.set_ylabel('Total de Contratos')
for i, v in enumerate(target_counts.values):
    ax1.text(i, v + 60, f"{v}\n({(v/len(df))*100:.1f}%)", ha='center', fontweight='bold')

# 2. Histograma do Valor Mensal
ax2 = plt.subplot(1, 3, 2)
sns.histplot(df['valor_mensal'].dropna(), kde=True, color='#2ca02c', bins=30, ax=ax2)
ax2.set_title('Gráfico 2: Distribuição do Valor Mensal', fontweight='bold')
ax2.set_xlabel('Valor Mensal (R$)')
ax2.set_ylabel('Frequência')

# 3. Matriz de Correlação de Pearson
ax3 = plt.subplot(1, 3, 3)
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='Blues', fmt='.2f', vmin=-1, vmax=1, ax=ax3, cbar=True)
ax3.set_title('Gráfico 3: Matriz de Correlação de Pearson', fontweight='bold')

plt.tight_layout()
plt.show()


### 🧠 Análise Interpretativa do Analista Sênior (EDA)

1. **Desbalanceamento Estrutural da Variável Alvo**:
   - Apenas **26,5%** da base cancelou o serviço (`cancelou = 1`), enquanto **73,5%** permaneceu ativa (`cancelou = 0`).
   - *Impacto na Modelagem*: Um classificador ingênuo (dummy classifier) que previsse todos os clientes como "ativos" obteria 73,5% de acurácia aparente, mas teria **Recall nulo (0%)** para o problema de negócio (não detectaria nenhuma evasão). O balanceamento de classes na etapa de treinamento é mandatório.

2. **Assimetria e Inconsistências nas Preditores**:
   - O atributo `valor_mensal` possui média de R$ 65,86, porém o valor máximo registrado alcança R$ 921,40 (coeficiente de assimetria / skewness > 7,8). Trata-se de um indicativo claro de anomalias/outliers provocados por falhas de integração dos sistemas legados.
   - A coluna `meses_contrato` varia de 0 a 72 meses, concentrando clientes recém-embarcados que exigirão tratamento preventivo contra divisão por zero na engenharia de variáveis.

3. **Correlações e Redundâncias**:
   - Forte correlação linear entre `meses_contrato` e `valor_total` ($r = 0,83$), o que reflete o acúmulo financeiro ao longo do ciclo de vida (LTV).
   - O atributo `id_cliente` é uma chave primária cadastral alfanumérica arbitrária: não possui relação causal com churn e geraria ruído ou memorização indevida se mantido.


## 2. Fase 2: Limpeza e Tratamento de Dados (Data Prep)

Nesta fase executamos:
1. Eliminação de registros duplicados oriundos de extrações redundantes.
2. Identificação e imputação de valores ausentes (`NaN`).
3. Diagnóstico visual de dispersão e outliers via Boxplots.


In [ ]:
# 1. Identificação e Remoção de Duplicatas
duplicados_qtd = df.duplicated().sum()
print(f"Total de registros duplicados detectados: {duplicados_qtd}")

df_clean = df.drop_duplicates().copy()
print(f"Dimensão pós-deduplicação: {df_clean.shape[0]} linhas.")

# 2. Diagnóstico de Nulos
nulos_resumo = df_clean.isnull().sum()[df_clean.isnull().sum() > 0]
print("\nAtributos com valores ausentes:")
print(nulos_resumo)


### Justificativa Técnica da Imputação por Mediana:
Optou-se estritamente pela **Mediana** em detrimento da Média pelas seguintes razões estatísticas:
* A **média aritmética** é linearmente sensível a valores extremos (outliers). Com valores de `valor_mensal` ultrapassando R$ 900,00 (onde 75% dos dados estão abaixo de R$ 89,95), a média inflacionaria artificialmente os registros imputados, introduzindo viés sistemático.
* A **mediana** é a estatística de ordem central robusta e invariante a assimetrias severas, preservando a distribuição original sem distorcer o espaço amostral.


In [ ]:
# Imputação robusta pela Mediana
cols_para_imputar = ['meses_contrato', 'valor_mensal', 'valor_total']

for col in cols_para_imputar:
    mediana_val = df_clean[col].median()
    df_clean[col].fillna(mediana_val, inplace=True)
    print(f"Imputação em '{col}': mediana aplicada = {mediana_val:.2f}")

# Validação de ausência de nulos residuais
print(f"\nTotal de nulos restantes nas colunas tratadas: {df_clean[cols_para_imputar].isnull().sum().sum()}")


In [ ]:
# 3. Geração de Boxplots para Diagnóstico de Outliers
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for idx, col in enumerate(cols_para_imputar):
    sns.boxplot(x=df_clean[col], ax=axes[idx], color='#9ecae1', fliersize=4)
    axes[idx].set_title(f'Boxplot: {col}', fontweight='bold')
    axes[idx].set_xlabel(col)

plt.tight_layout()
plt.show()


## 3. Fase 3: Engenharia de Recursos (Feature Engineering)

Cria-se a nova variável contínua `gasto_medio_mensal`, calculada pela relação entre o montante acumulado (`valor_total`) e o tempo de relacionamento (`meses_contrato`).

**Tratamento de Exceção (Divisão por Zero)**:  
Para clientes com `meses_contrato == 0` (novos entrantes no primeiro mês de ativação), a divisão matemática resultaria em indeterminação (`inf` / `NaN`). Nestes casos, aplica-se condicionalmente o próprio `valor_mensal` faturado.


In [ ]:
# Engenharia do atributo com proteção contra indeterminação matemática
df_clean['gasto_medio_mensal'] = np.where(
    df_clean['meses_contrato'] == 0,
    df_clean['valor_mensal'],
    df_clean['valor_total'] / df_clean['meses_contrato']
)

# Validação de integridade
print("Amostra do cálculo da nova variável:")
display(df_clean[['meses_contrato', 'valor_total', 'valor_mensal', 'gasto_medio_mensal']].head(6))

print(f"\nValores nulos ou infinitos em 'gasto_medio_mensal': {df_clean['gasto_medio_mensal'].isna().sum() + np.isinf(df_clean['gasto_medio_mensal']).sum()}")


## 4. Fase 4: Divisão dos Dados e Balanceamento

1. **Expurgo da Chave Primária**: remoção de `id_cliente`.
2. **Codificação Categórica**: aplicação de One-Hot Encoding (`pd.get_dummies`) com `drop_first=True` para suprimir colinearidade estrita entre categorias binárias e dummies correlacionadas.
3. **Divisão Treino/Teste (80/20)**: aplicação imperativa de `stratify=y` para garantir que a proporção original de cancelamentos (~26,5%) seja rigorosamente replicada em ambos os subconjuntos.
4. **Combate ao Data Leakage**: O algoritmo de sobreamostragem sintética (**SMOTE**) é instanciado e ajustado **estritamente sobre o conjunto de treinamento** (`X_train`, `y_train`). O conjunto de teste (`X_test`, `y_test`) permanece intacto com a distribuição real do negócio para avaliação fidedigna da generalização.


In [ ]:
# 1. Separação de Preditores e Target
X_raw = df_clean.drop(columns=['id_cliente', 'cancelou'])
y = df_clean['cancelou']

# 2. Codificação Dummy
X = pd.get_dummies(X_raw, drop_first=True)

# 3. Divisão Treino e Teste com Estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"Volume do Treino (Original): {X_train.shape[0]} registros")
print(f"Volume do Teste (Avaliação):  {X_test.shape[0]} registros")
print(f"Taxa de Churn no Treino: {y_train.mean():.4f}")
print(f"Taxa de Churn no Teste:  {y_test.mean():.4f}")

# 4. Aplicação do SMOTE apenas no Treino
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"\nDistribuição de Classes no Treino Pós-SMOTE:\n{pd.Series(y_train_bal).value_counts()}")


## 5. Fase 5: Escalonamento de Variáveis (StandardScaler)

### Fundamentação Teórico-Prática de Escala:
* **K-Nearest Neighbors (KNN)**:  
  O classificador KNN calcula a proximidade topológica entre instâncias através de métricas de distância geométrica (ex.: Distância Euclidiana: $d(p, q) = \sqrt{\sum (p_i - q_i)^2}$). Atributos em grandezas monetárias de milhares de reais (como `valor_total` $\in [18, 8684]$) dominariam completamente a métrica de distância sobre atributos de escala reduzida (como `meses_contrato` $\in [0, 72]$). Logo, a padronização via `StandardScaler` ($z = \frac{x - \mu}{\sigma}$) é indispensável.
  * *Prevenção de Vazamento*: O `StandardScaler` é ajustado via `fit_transform` exclusivamente em `X_train_bal` e aplicado via `transform` em `X_test`.

* **Árvore de Decisão**:  
  Algoritmos de árvore operam particionando o espaço de atributos recursivamente por meio de limiares univariados ortogonais ($X_j \le \theta$) baseados em métricas de impureza (Gini ou Entropia). Como qualquer transformação afim ou estritamente monótona preserva a ordem dos dados ($x_a < x_b \iff z_a < z_b$), a árvore é **completamente invariante à escala**. Treiná-la sobre os dados brutos preserva a semântica nativa das regras nos nós de decisão.


In [ ]:
cont_features = ['meses_contrato', 'valor_mensal', 'valor_total', 'gasto_medio_mensal']

scaler = StandardScaler()

# Cópia para isolar os pipelines de cada família de algoritmo
X_train_knn = X_train_bal.copy()
X_test_knn = X_test.copy()

# Ajuste no Treino e Transformação no Teste
X_train_knn[cont_features] = scaler.fit_transform(X_train_knn[cont_features])
X_test_knn[cont_features] = scaler.transform(X_test_knn[cont_features])

# Árvore de Decisão opera diretamente sobre os dados balanceados sem perda de interpretabilidade
X_train_tree = X_train_bal.copy()
X_test_tree = X_test.copy()

print("Escalonamento seletivo concluído com sucesso!")


## 6. Fase 6: Ajuste de Parâmetros e Combate ao Overfitting

Testamos múltiplos regimes de complexidade:
* **KNN**: Variando $K \in \{3, 5, 7\}$ (valores ímpares para evitar empates em classificação binária).
* **Árvore de Decisão**: Variando `max_depth` $\in \{3, 5, \text{None}\}$ (controlando a profundidade máxima e o crescimento irrestrito dos nós).


In [ ]:
metricas_treinamento = []

# Avaliação do KNN
for k in [3, 5, 7]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_knn, y_train_bal)
    
    acc_tr = accuracy_score(y_train_bal, knn.predict(X_train_knn))
    acc_te = accuracy_score(y_test, knn.predict(X_test_knn))
    
    metricas_treinamento.append({
        'Algoritmo': 'KNN',
        'Configuração': f'n_neighbors={k}',
        'Acurácia Treino': acc_tr,
        'Acurácia Teste': acc_te,
        'Delta (|Treino - Teste|)': abs(acc_tr - acc_te)
    })

# Avaliação da Árvore de Decisão
for d in [3, 5, None]:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train_tree, y_train_bal)
    
    acc_tr = accuracy_score(y_train_bal, dt.predict(X_train_tree))
    acc_te = accuracy_score(y_test, dt.predict(X_test_tree))
    
    metricas_treinamento.append({
        'Algoritmo': 'Árvore de Decisão',
        'Configuração': f'max_depth={d}',
        'Acurácia Treino': acc_tr,
        'Acurácia Teste': acc_te,
        'Delta (|Treino - Teste|)': abs(acc_tr - acc_te)
    })

df_comparativo = pd.DataFrame(metricas_treinamento)
display(df_comparativo.style.format({
    'Acurácia Treino': '{:.2%}',
    'Acurácia Teste': '{:.2%}',
    'Delta (|Treino - Teste|)': '{:.2%}'
}))


In [ ]:
# Visualização gráfica do trade-off Viés-Variância (Overfitting)
fig, ax = plt.subplots(figsize=(10, 4))
df_plot = df_comparativo.copy()
df_plot['Label'] = df_plot['Algoritmo'] + " (" + df_plot['Configuração'] + ")"

x = np.arange(len(df_plot))
width = 0.35

rects1 = ax.bar(x - width/2, df_plot['Acurácia Treino'], width, label='Treino', color='#4575b4')
rects2 = ax.bar(x + width/2, df_plot['Acurácia Teste'], width, label='Teste', color='#d73027')

ax.set_ylabel('Acurácia')
ax.set_title('Diagnóstico de Generalização vs Overfitting', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_plot['Label'], rotation=25, ha='right')
ax.set_ylim(0.5, 1.05)
ax.legend()
plt.tight_layout()
plt.show()


### 🔬 Diagnóstico Analítico de Overfitting:
1. **Árvore de Decisão sem Restrição (`max_depth=None`)**:  
   Exibe o caso canônico de **sobreajuste severo (overfitting)**. O modelo atinge **99,77%** de acurácia no treino ao subdividir o espaço amostral até que as folhas fiquem puras, memorizando os ruídos estatísticos e o ruído sintético gerado pelo SMOTE. Ao ser exposto aos dados de teste inéditos, sua acurácia desaba para **67,21%** (delta de ~32,5% de degradação).
2. **Poda e Regularização (`max_depth=3`)**:  
   Ao restringir a árvore a 3 níveis de profundidade, limitamos a variância do estimador. A acurácia de treino ajusta-se para 71,97% e a acurácia de teste sobe para **75,09%**, demonstrando excelente capacidade de generalização sem memorização.
3. **Comportamento do KNN**:  
   Para $K=3$, o modelo é hiper-reativo a vizinhos individuais ruidosos (Treino 83,98%, Teste 70,55%). O aumento para $K=7$ suaviza a superfície de decisão estatística, elevando o teste para 71,54%.


## 7. Fase 7: Avaliação da Acurácia e Veredito Final de Negócio

Extraímos o desempenho final em teste dos dois melhores concorrentes ajustados:
- **Melhor KNN**: $K = 7$
- **Melhor Árvore de Decisão**: `max_depth = 3`


In [ ]:
# Treinamento e Predição dos Melhores Modelos Selecionados
modelo_knn_final = KNeighborsClassifier(n_neighbors=7)
modelo_knn_final.fit(X_train_knn, y_train_bal)
y_pred_knn = modelo_knn_final.predict(X_test_knn)
acc_knn_final = accuracy_score(y_test, y_pred_knn)

modelo_tree_final = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo_tree_final.fit(X_train_tree, y_train_bal)
y_pred_tree = modelo_tree_final.predict(X_test_tree)
acc_tree_final = accuracy_score(y_test, y_pred_tree)

print("="*65)
print("           RELATÓRIO COMPARATIVO FINAL - CONJUNTO DE TESTE")
print("="*65)
print(f"1. K-Nearest Neighbors (K=7):          Acurácia = {acc_knn_final:.4f} ({acc_knn_final*100:.2f}%)")
print(f"2. Árvore de Decisão (max_depth=3):    Acurácia = {acc_tree_final:.4f} ({acc_tree_final*100:.2f}%)")
print("="*65)

# Matriz de Confusão Comparativa
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_knn), annot=True, fmt='d', cmap='Blues', ax=ax1, cbar=False)
ax1.set_title(f'KNN (K=7) - Acc: {acc_knn_final:.2%}')
ax1.set_xlabel('Predito')
ax1.set_ylabel('Real')

sns.heatmap(confusion_matrix(y_test, y_pred_tree), annot=True, fmt='d', cmap='Greens', ax=ax2, cbar=False)
ax2.set_title(f'Árvore (Prof=3) - Acc: {acc_tree_final:.2%}')
ax2.set_xlabel('Predito')
ax2.set_ylabel('Real')

plt.tight_layout()
plt.show()


### 🏆 Veredito Final e Recomendação Estratégica do Analista Sênior

O modelo **Árvore de Decisão com poda de profundidade (`max_depth=3`)** é o algoritmo formalmente selecionado para implantação em produção pela companhia.

#### Justificativas Técnicas e Estratégicas:
1. **Superioridade Métrica no Conjunto de Teste**:
   - A Árvore alcançou **75,09%** de acurácia, superando o KNN com $K=7$ (**71,54%**), comprovando maior robustez em dados reais não vistos.
2. **Interpretabilidade e Acionabilidade Executiva (White-Box)**:
   - Uma árvore com profundidade 3 gera regras determinísticas e auditáveis (ex.: *se contrato mensal e tempo < 12 meses, risco crítico*). Essas regras podem ser convertidas instantaneamente em gatilhos automatizados no CRM da equipe de marketing. O KNN opera como uma caixa preta de cálculo de vizinhança geométrica no espaço de alta dimensão, dificultando a explicação do motivo do churn ao comitê de negócios.
3. **Latência de Inferência e Escalabilidade Operacional**:
   - **Árvore de Decisão**: Avalia uma nova transação em tempo logarítmico $\mathcal{O}(\text{profundidade})$, exigindo apenas 3 comparações lógicas condicionais simples.
   - **KNN**: Classificador baseado em memória (lazy learner) que calcula distâncias contra toda a base histórica em $\mathcal{O}(N \cdot D)$. Em escala de milhões de assinantes, o KNN gera gargalos severos de memória e latência.
